# Lekcija 19: Grupiranje — k-means, GMM i DBSCAN

Sve prethodne lekcije u ovom dijelu pretpostavljale su neki *cilj* — prag, predložak ili poznatu transformaciju. **Grupiranje** (engl. *clustering*) postavlja drukčije pitanje, bez unaprijed zadanog cilja: za skup točaka ili piksela pronaći mali broj smislenih skupina koristeći samo njihovu međusobnu sličnost. To je klasično polazište **nenadziranog učenja** — bez oznaka, samo uz funkciju udaljenosti i pretpostavku o tome kako bi skupina trebala izgledati. U ovoj lekciji od početka implementiramo tri algoritma grupiranja, svaki s drukčijom pretpostavkom o obliku skupine, te konkretno pokazujemo gdje uspijevaju, a gdje ne: **k-means**, za približno kružne skupine slične veličine, **modele Gaussovih mješavina**, za eliptične skupine i meko pridruživanje, te **DBSCAN**, za proizvoljne oblike definirane gustoćom umjesto središtem.


In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt

## k-means na stvarnim podacima o boji

U Lekciji 18 izgradili smo perceptivno smislenu udaljenost boja: euklidsku udaljenost u L\*a\*b\* prostoru. Grupiranje je prirodan sljedeći korak: iz skupa boja piksela bez ikakvih oznaka automatski odrediti područja. Izradite umjetnu sliku u boji s 4 područja i šumom, kako se ne bi mogla trivijalno razdvojiti točnim podudaranjem boje, pretvorite je u L*a*b* i grupirajte vrijednosti boja piksela.


In [ ]:
SIZE = 48

def make_region_image(size=SIZE, noise_std=8, seed=3):
    rng = np.random.default_rng(seed)
    img = np.zeros((size, size, 3), dtype=np.uint8)
    region_map = np.zeros((size, size), dtype=np.int64)
    colors_bgr = [(180, 40, 30), (40, 160, 60), (30, 80, 190), (200, 200, 60)]
    centers = [(12, 12), (12, 36), (36, 12), (36, 36)]
    yy, xx = np.mgrid[0:size, 0:size]
    for k, ((cy, cx), col) in enumerate(zip(centers, colors_bgr)):
        mask = (xx - cx) ** 2 + (yy - cy) ** 2 <= 20 ** 2
        img[mask] = col
        region_map[mask] = k + 1
    noise = rng.normal(0, noise_std, img.shape)
    img = np.clip(img.astype(np.float64) + noise, 0, 255).astype(np.uint8)
    return img, region_map

img, region_map = make_region_image()
lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB).astype(np.float64)
X_pixels = lab.reshape(-1, 3)

def kmeans(X, k, n_iter=50, seed=0):
    rng = np.random.default_rng(seed)
    centers = X[rng.choice(len(X), k, replace=False)].copy()
    for _ in range(n_iter):
        dists = ((X[:, None, :] - centers[None, :, :]) ** 2).sum(-1)
        labels = dists.argmin(1)
        new_centers = np.array([X[labels == j].mean(0) if (labels == j).any() else centers[j]
                                 for j in range(k)])
        if np.allclose(new_centers, centers):
            break
        centers = new_centers
    return labels, centers

def purity(labels, true_flat):
    correct = 0
    for u in np.unique(labels):
        mask = labels == u
        majority = np.bincount(true_flat[mask]).argmax()
        correct += (true_flat[mask] == majority).sum()
    return correct / len(true_flat)

km_labels, _ = kmeans(X_pixels, k=4, seed=3)
km_seg = km_labels.reshape(SIZE, SIZE)
print(f'k-means segmentation purity vs. true regions: {purity(km_labels, region_map.ravel()):.3f}')

fig, axes = plt.subplots(1, 3, figsize=(9, 3.2))
axes[0].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)); axes[0].set_title('input (noisy)', fontsize=9); axes[0].axis('off')
axes[1].imshow(region_map, cmap='tab10'); axes[1].set_title('true regions', fontsize=9); axes[1].axis('off')
axes[2].imshow(km_seg, cmap='tab10'); axes[2].set_title('k-means segmentation', fontsize=9); axes[2].axis('off')
plt.show()

K-means, poznat i kao Lloydov algoritam, ovdje je u svom standardnom obliku: odaberite `k` nasumičnih točaka kao početna središta, svaku točku pridružite najbližem središtu, ponovno izračunajte svako središte kao prosjek pridruženih točaka i ponavljajte dok se ništa više ne mijenja. Za kompaktne, približno jednake i približno kružne skupine u L*a*b* prostoru, kakve upravo ima ova umjetna slika, gotovo savršeno pronalazi stvarna područja. I u ovom uspješnom primjeru postoje dva važna ograničenja: `k`, odnosno broj skupina, mora se odabrati unaprijed jer ga algoritam ne otkriva, a različite nasumične inicijalizacije mogu konvergirati prema različitim, ponekad znatno lošijim lokalnim optimumima. Isprobajte promjenu `seed=3` u `seed=0` u prethodnom primjeru i ponovno pokrenite algoritam. **K-means++** (<a href="../references.html#arthur-2007-kmeanspp">Arthur & Vassilvitskii, 2007</a><span class="landmark-paper">&#9733;</span>) to rješava odabirom međusobno udaljenih početnih središta umjesto potpuno nasumičnog odabira.


## Kada k-means ne uspijeva: nekonveksne skupine

Pravilo ažuriranja k-meansa — pridruživanje najbližem *središtu* — može podijeliti prostor samo na konveksna područja. Izradimo skup podataka koji uopće nije konveksan: dva koncentrična prstena. Nijedan položaj dvaju središta ne može ih ispravno razdvojiti, bez obzira na broj iteracija algoritma.


In [ ]:
rng = np.random.default_rng(1)

def ring(n, r, noise=0.05):
    theta = rng.uniform(0, 2 * np.pi, n)
    rad = r + rng.normal(0, noise, n)
    return np.stack([rad * np.cos(theta), rad * np.sin(theta)], axis=1)

X_rings = np.vstack([ring(150, 1.0), ring(150, 2.5)])
true_rings = np.array([0] * 150 + [1] * 150)

def best_binary_acc(labels, true):
    return max((labels == true).mean(), (labels == (1 - true)).mean())

km_ring_labels, km_ring_centers = kmeans(X_rings, k=2, seed=0)
print(f'k-means accuracy on concentric rings: {best_binary_acc(km_ring_labels, true_rings):.1%}  (2-class chance = 50%)')

fig, axes = plt.subplots(1, 2, figsize=(7, 3.5))
axes[0].scatter(*X_rings.T, c=true_rings, cmap='coolwarm', s=8)
axes[0].set_title('true rings', fontsize=9); axes[0].set_aspect('equal')
axes[1].scatter(*X_rings.T, c=km_ring_labels, cmap='coolwarm', s=8)
axes[1].scatter(*km_ring_centers.T, c='black', marker='x', s=100, label='centers')
axes[1].set_title('k-means result', fontsize=9); axes[1].set_aspect('equal'); axes[1].legend(fontsize=7)
plt.show()

K-means postiže rezultat na razini slučajnog odabira — radijalno dijeli dva prstena, pri čemu polovica svakog prstena pripada svakoj skupini, jer je jedina granica koju algoritam može povući pravac jednako udaljen od dvaju središta. Ljudskom oku ove skupine uopće nisu nejasne; *pretpostavka algoritma* o kružnim skupinama temeljenima na središtu jednostavno ne odgovara ovom obliku.


## Modeli Gaussovih mješavina: meke, eliptične skupine

**Model Gaussove mješavine (GMM)** generalizira k-means na dva načina: svaka skupina dobiva vlastitu punu matricu kovarijance, pa može biti izdužena elipsa, a ne samo krug, i pridruživanje je *meko* — svaka točka dobiva vjerojatnost pripadnosti svakoj skupini umjesto jedne čvrste oznake. Model se prilagođava postupkom **očekivanja i maksimizacije (EM)** (<a href="../references.html#dempster-1977-em">Dempster, Laird, & Rubin, 1977</a><span class="landmark-paper">&#9733;</span>): E-korak izračunava odgovornost, odnosno vjerojatnost pripadnosti svake točke, prema trenutačnim Gaussovim komponentama; M-korak ponovno procjenjuje srednju vrijednost, kovarijancu i težinu svake komponente pomoću točaka ponderiranih tim odgovornostima.


In [ ]:
def gmm_em(X, k, n_iter=100, seed=0):
    rng = np.random.default_rng(seed)
    n, d = X.shape
    means = X[rng.choice(n, k, replace=False)].copy()
    covs = np.array([np.cov(X.T) + 1e-3 * np.eye(d) for _ in range(k)])
    weights = np.full(k, 1 / k)
    for _ in range(n_iter):
        resp = np.zeros((n, k))
        for j in range(k):
            diff = X - means[j]
            inv = np.linalg.inv(covs[j])
            expo = -0.5 * np.sum(diff @ inv * diff, axis=1)
            norm = 1.0 / np.sqrt((2 * np.pi) ** d * np.linalg.det(covs[j]))
            resp[:, j] = weights[j] * norm * np.exp(expo)
        resp /= resp.sum(1, keepdims=True) + 1e-12
        Nk = resp.sum(0)
        weights = Nk / n
        means = (resp.T @ X) / Nk[:, None]
        for j in range(k):
            diff = X - means[j]
            covs[j] = (resp[:, j:j + 1] * diff).T @ diff / Nk[j] + 1e-6 * np.eye(d)
    return resp.argmax(1), means, covs

gmm_ring_labels, gmm_means, gmm_covs = gmm_em(X_rings, k=2, seed=0)
print(f'GMM accuracy on concentric rings: {best_binary_acc(gmm_ring_labels, true_rings):.1%}')

GMM ovdje ne daje bolji rezultat od k-meansa — također je blizu slučajnom odabiru. To je važan detalj koji se lako previdi: GMM uklanja pretpostavku k-meansa o *isključivo kružnim* skupinama, ali i dalje koristi **jednu unimodalnu distribuciju po skupini** — svaku skupinu opisuje jedna Gaussova izbočina. Prsten nije eliptičan ništa više nego što je kružan u tom smislu; nijedna pojedinačna Gaussova distribucija, bez obzira na oblik, ne opisuje dobro prstenasto područje. GMM i k-means na ovom skupu podataka ne uspijevaju iz istog temeljnog razloga.


## DBSCAN: skupine definirane gustoćom, a ne oblikom

**DBSCAN**, prostorno grupiranje temeljeno na gustoći (<a href="../references.html#ester-1996-dbscan">Ester i sur., 1996</a><span class="landmark-paper">&#9733;</span>), uopće ne prilagođava parametarski oblik. Točka je **jezgrena točka** ako se najmanje `min_samples` drugih točaka nalazi na udaljenosti do `eps` od nje. Skupine nastaju povezivanjem jezgrenih točaka međusobno udaljenih najviše `eps`, zajedno s njihovim susjedima, pa skupina može imati bilo koji oblik, uključujući prsten, ako čini *povezano, dovoljno gusto* područje prostora. Točke koje se ne nalaze u susjedstvu nijedne jezgrene točke označavaju se kao **šum**, umjesto prisilnog pridruživanja najbližoj skupini.


In [ ]:
def dbscan(X, eps, min_samples):
    n = len(X)
    labels = np.full(n, -1)
    visited = np.zeros(n, dtype=bool)
    dist = np.sqrt(((X[:, None, :] - X[None, :, :]) ** 2).sum(-1))
    cluster_id = 0
    for i in range(n):
        if visited[i]:
            continue
        visited[i] = True
        neighbors = list(np.where(dist[i] <= eps)[0])
        if len(neighbors) < min_samples:
            continue  # stays noise (-1)
        labels[i] = cluster_id
        j = 0
        while j < len(neighbors):
            q = neighbors[j]
            if not visited[q]:
                visited[q] = True
                q_neighbors = np.where(dist[q] <= eps)[0]
                if len(q_neighbors) >= min_samples:
                    neighbors.extend([x for x in q_neighbors if x not in neighbors])
            if labels[q] == -1:
                labels[q] = cluster_id
            j += 1
        cluster_id += 1
    return labels

db_ring_labels = dbscan(X_rings, eps=0.6, min_samples=4)
n_clusters = len(set(db_ring_labels.tolist()) - {-1})
n_noise = (db_ring_labels == -1).sum()
print(f'DBSCAN found {n_clusters} clusters, {n_noise} noise points')
print(f'DBSCAN accuracy on concentric rings: {best_binary_acc(db_ring_labels, true_rings):.1%}')

fig, axes = plt.subplots(1, 3, figsize=(10, 3.5))
for ax, labels, title in zip(axes, [true_rings, km_ring_labels, db_ring_labels],
                              ['true rings', 'k-means (fails)', 'DBSCAN (succeeds)']):
    ax.scatter(*X_rings.T, c=labels, cmap='coolwarm', s=8)
    ax.set_title(title, fontsize=9); ax.set_aspect('equal')
plt.show()

DBSCAN savršeno razdvaja dva prstena — ne pretpostavlja središte ni oblik, nego samo da su točke *unutar* prstena gusto povezane sa susjedima, dok razmak između prstena nije. To ipak ima cijenu: `eps` i `min_samples` moraju se odabrati ručno, a loš odabir razbija svaki prsten na desetke malih lukova — isprobajte `eps=0.15` u prethodnom primjeru da biste to vidjeli. DBSCAN ima poteškoće i kada različite stvarne skupine imaju vrlo različite gustoće jer jedna globalna vrijednost `eps` ne može istodobno odgovarati i rijetkoj i gustoj skupini.


## Otpornost na šum: prednost izostanka prisilnog pridruživanja

Dodajte ometajuće točke jednoliko raspršene po cijelom području, koje ne pripadaju nijednom prstenu, i usporedite kako ih svaki algoritam obrađuje. K-means i GMM nemaju pojam „ne pripada nijednoj skupini”: svaka se točka nekamo pridružuje, povlačeći središta skupina prema ometajućim točkama. DBSCAN ih može jednostavno označiti onim što jesu.


In [ ]:
noise_pts = rng.uniform(-3, 3, (60, 2))
X_cluttered = np.vstack([X_rings, noise_pts])
true_cluttered = np.concatenate([true_rings, np.full(60, -1)])  # -1 = "not really a ring"

km_clut_labels, _ = kmeans(X_cluttered, k=2, seed=0)
db_clut_labels = dbscan(X_cluttered, eps=0.6, min_samples=4)

km_noise_correctly_flagged = 0  # k-means has no noise concept at all
db_noise_correctly_flagged = (db_clut_labels[300:] == -1).sum()

print(f'{"":>10} {"noise points correctly flagged":>32} {"out of":>8}')
print(f'{"k-means":>10} {km_noise_correctly_flagged:>32} {60:>8}  (no noise concept — every point forced into a cluster)')
print(f'{"DBSCAN":>10} {db_noise_correctly_flagged:>32} {60:>8}')

K-means po svojoj konstrukciji nijednu od 60 ometajućih točaka ne označava drukčije od „sigurno dio prstena” — algoritam nema kategoriju „ne pripada nikamo”. DBSCAN ispravno izdvaja manji dio njih; ostale slučajno padaju unutar udaljenosti `eps` od stvarne točke prstena ili jedna od druge, stvarajući vlastitu malu skupinu. To je očekivan rezultat, a ne pogreška — grupiranje prema gustoći označava točku kao šum samo ako je doista izolirana, a jednoliko nasumično raspršivanje ipak povremeno daje nakupine. Glavna poruka vrijedi bez obzira na točan broj: DBSCAN ima stvarnu, upotrebljivu kategoriju „ovo se nigdje ne uklapa”, dok je k-means po svojoj strukturi nema.


## Što slijedi

Ovo je kraj prvog dijela. Mnogi dosad obrađeni algoritmi rade izravno s **izvornim ili ručno odabranim značajkama** — bojom piksela, koordinatama `(x, y)` i teksturom — te se oslanjaju na konkretna pravila, poput ovdje korištenih funkcija udaljenosti koje je odabrao čovjek. U sljedećim lekcijama mnoge ćemo probleme ponovno razmotriti uz duboko učenje, koje automatski otkriva značajke i pravila njihova kombiniranja. Primjerice, segmentacija pojedinačnih objekata iz Lekcije 43 koristi sličnu ideju grupiranja, a autoenkoder iz Lekcije 48 uči sam prostor značajki samo iz pogreške rekonstrukcije, prije bilo kakva grupiranja. Dok metode ove lekcije pretpostavljaju oblik i prilagođavaju ga podacima, metode dubokog učenja u nastavku kolegija uče reprezentaciju iz koje struktura proizlazi.

### Zadaci

1. Ponovno pokrenite k-means segmentaciju boje s `k=3` i `k=6` umjesto stvarnog broja `4`. Budući da algoritam ne može „znati” točan broj skupina, kako se segmentacija pogoršava u svakom smjeru i možete li samo iz izlaza zaključiti koji je `k` vjerojatno pogrešan?
2. Prethodni GMM vrednovan je samo na skupu prstenova, gdje ne uspijeva iz istog razloga kao k-means. Izradite skup podataka s dvjema doista eliptičnim skupinama, primjerice pomoću `np.random.multivariate_normal` s nedijagonalnom matricom kovarijance, i ondje usporedite GMM s k-meansom. Daje li dodatna fleksibilnost GMM-a, koji prilagođava oblik, a ne samo središte, ovaj put stvarnu prednost u točnosti?
3. U DBSCAN primjeru s ometajućim šumom smanjite `eps` s `0.6` na `0.3`. Počinje li DBSCAN pogrešno označavati stvarne točke prstena kao šum, ometajuće točke kao dio prstena ili oboje? Odgovara li to prethodnom upozorenju da je `eps` ručno podešen odabir specifičan za skup podataka, a ne nešto što algoritam sam otkriva?
